# CodeTune v3.1 — Colab runbook (A100/H100)

One Colab session = one stage. Flip the `RUN_*` switches in the **Config** cell, then *Run all*.
Everything that matters is written to Google Drive as it is produced, so a disconnect only costs the current step
(SFT resumes automatically from the newest checkpoint).

| Session | Stage | Switches to turn on |
|---|---|---|
| 1 | Baseline evaluation (+ trust gate) | `RUN_EVAL_BASE` |
| 2 | SFT (aggressive + conservative) and their evaluation | `RUN_SFT`, `RUN_EVAL_SFT` |
| 3 | Preference pairs, DPO (β=0.1 / 0.3) and evaluation | `RUN_DPO_PAIR_GEN`, `RUN_DPO`, `RUN_EVAL_DPO` |

Plan: `report/CodeTune_v3.1_Execution_Plan.md`. Do not compare any numbers until the baseline **trust gate** passes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], check=True)
# Do not import torch here: the Environment cell may replace both torch and torchvision.


## Config

In [ ]:
import datetime

# --- Repo -------------------------------------------------------------------
REPO_URL = 'https://github.com/MichLitt/coding-llm-finetune.git'
REPO_BRANCH = 'fix/v3.1-eval-harness'   # switch to 'main' after the branch is merged
WORKDIR = '/content/Coding-LLM'

# --- Google Drive layout -----------------------------------------------------
DRIVE_ROOT = '/content/drive/MyDrive/coding-llm-colab'
DATA_SOURCE_DIR = '/content/drive/MyDrive/coding-llm-data-v3.1/processed'   # sft_train.jsonl / sft_val.jsonl
ARTIFACT_ROOT = f'{DRIVE_ROOT}/artifacts'   # checkpoints + eval results (written live, shared across sessions)
RUN_TAG = 'baseline'                        # free text; names this session's run manifest
RUN_DIR = f"{DRIVE_ROOT}/runs/{datetime.date.today().isoformat()}_{RUN_TAG}"

# --- Secrets: paste only inside the Colab session, never commit ------------------
HF_TOKEN = ''
WANDB_API_KEY = ''

# --- Stage switches -----------------------------------------------------------
RUN_EVAL_BASE = False
RUN_SFT = False
RUN_EVAL_SFT = False
RUN_DPO_PAIR_GEN = False
RUN_DPO = False
RUN_EVAL_DPO = False

# --- Experiments ----------------------------------------------------------------
BASE_MODEL = 'unsloth/Qwen3.5-4B'                 # mirror of Qwen/Qwen3.5-4B (post-trained release)
SFT_EXP_IDS = ['sft_generic_aggr', 'sft_generic_cons']
# Starting point for preference-pair generation and DPO: 'base' or an SFT adapter dir on Drive,
# e.g. f'{ARTIFACT_ROOT}/results/sft_checkpoints/sft_generic_cons/final'.
# Decision rule (plan stage 2): use the best SFT adapter if it is not worse than base, otherwise 'base'.
DPO_START = 'base'
DPO_EXPS = {'dpo_b01': 0.1, 'dpo_b03': 0.3}       # exp id -> beta
DPO_PAIRS_DIR = f'{ARTIFACT_ROOT}/data/dpo_pairs'  # pairs are copied here after generation
MBPP_N_CANDIDATES = 8

# --- Evaluation ---------------------------------------------------------------
EVAL_DATASETS = ['humaneval', 'mbpp']   # HumanEval(+) and MBPP+
EVAL_BACKEND = 'unsloth'                # 'unsloth' (verified path) | 'hf' | 'vllm' (experimental)

## Environment

Versions are **not pinned yet**. After the first fully successful run, copy the versions from
`<RUN_DIR>/pip_freeze.txt` into `PIP_PINS` below so later sessions reproduce the same stack.

In [ ]:
import subprocess, sys

# Fill after the first successful run, e.g. ['trl==0.x.y', 'peft==0.x.y', 'evalplus==0.3.1'].
PIP_PINS = []

# Let Unsloth select one matching GPU torch/torchvision backend for this Colab runtime.
# Do not install either package separately: partial upgrades cause ABI errors such as
# `operator torchvision::nms does not exist`.
%pip install -q --upgrade uv
subprocess.run(['uv', 'pip', 'install', '--system', '--upgrade', 'unsloth', '--torch-backend=auto'], check=True)
%pip install -q trl peft accelerate bitsandbytes datasets huggingface-hub python-dotenv pyyaml click wandb "evalplus>=0.3.1"
if PIP_PINS:
    %pip install -q {' '.join(PIP_PINS)}

import torch, torchvision
print('torch:', torch.__version__, '| torchvision:', torchvision.__version__)

## Workspace: clone repo, sync data, write run manifest

In [ ]:
import json, os, shutil, subprocess
from pathlib import Path

workdir = Path(WORKDIR)
if workdir.exists():
    shutil.rmtree(workdir)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(workdir)], check=True)
GIT_COMMIT = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=workdir, capture_output=True, text=True).stdout.strip()

for sub in (RUN_DIR, ARTIFACT_ROOT):
    Path(sub).mkdir(parents=True, exist_ok=True)

if Path(DATA_SOURCE_DIR).exists():
    (workdir / 'data/processed').mkdir(parents=True, exist_ok=True)
    subprocess.run(['rsync', '-a', f'{DATA_SOURCE_DIR}/', str(workdir / 'data/processed') + '/'], check=True)
else:
    print('WARNING: DATA_SOURCE_DIR missing (fine for the baseline-only session):', DATA_SOURCE_DIR)

env_lines = [f'{k}={v}' for k, v in (('HF_TOKEN', HF_TOKEN), ('WANDB_API_KEY', WANDB_API_KEY)) if v]
if env_lines:
    (workdir / '.env').write_text('\n'.join(env_lines) + '\n')

# Checkpoints and eval results are written straight to Drive (CODETUNE_OUTPUT_ROOT), so nothing is lost on disconnect.
os.environ['CODETUNE_OUTPUT_ROOT'] = ARTIFACT_ROOT
os.environ['PYTHONUNBUFFERED'] = '1'

gpu = torch.cuda.get_device_name(0)
profile = 'h100' if 'H100' in gpu.upper() else 'a100'
SFT_CONFIG_PATH = f'configs/sft_config_colab_{profile}.yaml'
DPO_CONFIG_PATH = f'configs/dpo_config_colab_{profile}.yaml'

freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout
Path(RUN_DIR, 'pip_freeze.txt').write_text(freeze)
Path(RUN_DIR, 'manifest.json').write_text(json.dumps({
    'git_commit': GIT_COMMIT, 'repo_branch': REPO_BRANCH, 'gpu': gpu, 'cuda': torch.version.cuda,
    'torch': torch.__version__, 'profile': profile, 'sft_config': SFT_CONFIG_PATH, 'dpo_config': DPO_CONFIG_PATH,
    'switches': {k: v for k, v in globals().items() if k.startswith('RUN_') and isinstance(v, bool)},
    'base_model': BASE_MODEL, 'dpo_start': DPO_START,
    'sft_config_text': Path(workdir, SFT_CONFIG_PATH).read_text(),
    'ablation_matrix': Path(workdir, 'configs/ablation_matrix.yaml').read_text(),
}, indent=2))
print('commit', GIT_COMMIT, '| profile', profile, '| manifest ->', RUN_DIR)

## Helpers

In [ ]:
import sys
sys.path.insert(0, str(Path(WORKDIR) / 'scripts'))

def run(cmd):
    cmd = ['python', '-u'] + cmd[1:] if cmd[0] == 'python' else cmd
    print('$', ' '.join(cmd))
    proc = subprocess.Popen(cmd, cwd=WORKDIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    log = Path(RUN_DIR, 'log.txt').open('a')
    try:
        for line in proc.stdout:
            print(line, end=''); log.write(line)
        if proc.wait() != 0:
            raise subprocess.CalledProcessError(proc.returncode, cmd)
    finally:
        proc.stdout.close(); log.close()

EVAL_ROOT = f'{ARTIFACT_ROOT}/results/eval'

def evaluate_model(label, model):
    """Same command, same flags for every model (base / SFT / DPO)."""
    for ds in EVAL_DATASETS:
        run(['python', 'scripts/run_eval.py', '--model', model, '--label', label, '--dataset', ds,
             '--backend', EVAL_BACKEND, '--results-dir', EVAL_ROOT])

def show_summary(label):
    from run_eval import eval_gate
    for ds in EVAL_DATASETS:
        path = Path(EVAL_ROOT, label, ds, 'summary.json')
        if not path.exists():
            continue
        s = json.loads(path.read_text())
        print(f"{label:>18} {ds:>9}: pass@1 {s['pass_at_1']:.1%} ({s['passed']}/{s['total']}) | "
              f"plus {s.get('evalplus_plus_pass_at_1', float('nan')):.1%} | hit-limit {s['hit_token_limit_rate']:.1%} | {s['status_counts']}")
        gate = eval_gate(s)
        print('   trust gate:', 'PASS' if all(gate.values()) else 'FAIL', gate)

## Stage 1 — baseline evaluation and trust gate

In [ ]:
if RUN_EVAL_BASE:
    evaluate_model('base', BASE_MODEL)
    show_summary('base')
    print('\nIf any trust gate FAILED: stop. Inspect results/eval/base/*/raw.jsonl and per_problem.jsonl '
          '(truncation? extraction?) and fix the harness before training anything.')

## Stage 2 — SFT (resumes automatically after a disconnect)

In [ ]:
if RUN_SFT:
    for exp_id in SFT_EXP_IDS:
        run(['python', 'scripts/sft_train.py', '--config-path', SFT_CONFIG_PATH, '--exp-id', exp_id, '--resume-from', 'auto'])

if RUN_EVAL_SFT:
    for exp_id in SFT_EXP_IDS:
        evaluate_model(exp_id, f'{ARTIFACT_ROOT}/results/sft_checkpoints/{exp_id}/final')
        show_summary(exp_id)
    print('\nDecision rule: compare each SFT run with base (paired, see scripts/compare_runs.py). '
          'If both regress, set DPO_START = "base".')

## Stage 3 — preference pairs, DPO, evaluation

In [ ]:
if RUN_DPO_PAIR_GEN:
    run(['python', 'scripts/generate_dpo_pairs.py', '--sft-checkpoint', DPO_START,
         '--n-candidates', str(MBPP_N_CANDIDATES), '--backend', EVAL_BACKEND])
    Path(DPO_PAIRS_DIR).mkdir(parents=True, exist_ok=True)
    subprocess.run(['rsync', '-a', f'{WORKDIR}/data/processed/', f'{DPO_PAIRS_DIR}/'], check=True)
    print('pairs saved to', DPO_PAIRS_DIR)
    print(Path(WORKDIR, 'data/processed/dpo_pairs_stats.json').read_text())
    print('\nStop here if the "Stage-3 gate" line above contains a False. Do not train DPO on rejected pairs.')

if RUN_DPO:
    if Path(DPO_PAIRS_DIR).exists():
        subprocess.run(['rsync', '-a', f'{DPO_PAIRS_DIR}/', f'{WORKDIR}/data/processed/'], check=True)
    for exp_id, beta in DPO_EXPS.items():
        run(['python', 'scripts/dpo_train.py', '--config-path', DPO_CONFIG_PATH, '--sft-checkpoint', DPO_START,
             '--exp-id', exp_id, '--beta', str(beta)])

if RUN_EVAL_DPO:
    for exp_id in DPO_EXPS:
        evaluate_model(exp_id, f'{ARTIFACT_ROOT}/results/dpo_checkpoints/{exp_id}/final')
        show_summary(exp_id)

## Wrap-up

Results live in `<ARTIFACT_ROOT>/results/eval/<label>/<dataset>/`. Download that folder (summaries and
per-problem files only, not weights) to the local repo and run the paired comparison on your Mac.

In [ ]:
for label in ['base', *SFT_EXP_IDS, *DPO_EXPS]:
    show_summary(label)